In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Tip: For large simulations, switch to a GPU runtime first:
#   Runtime → Change runtime type → T4 GPU (free)
# Then re-run this cell to install the CUDA-enabled JAX.
try:
    import google.colab  # noqa: F401
    import shutil

    if shutil.which("nvidia-smi"):
        %pip install -q -U softmobility "jax[cuda12]"
    else:
        %pip install -q softmobility
except ImportError:
    pass

# Tutorial 03. Soft mobility tensors and simulation of a trajectory

A *soft body* in SoftMobility differs from a rigid one by carrying **internal degrees of freedom** (DOFs) that the solver integrates in time alongside the body position and orientation.
Each DOF is generally paired with an elastic restoring force/torque (and optional input couplings), and the framework automatically derives the soft mobility tensors

$$\mathbf{M},\ \mathbf{M}_K,\ \mathbf{M}_H,\ \mathbf{C}_E,\ \boldsymbol{\Pi}$$

that map external loads, DOF deformation, and ambient flow strain to the time derivatives of the body coordinates and DOFs.

This tutorial:

1. builds a one-DOF deformable dumbbell with a torsion spring,
2. inspects the soft mobility tensors at the default configuration,
3. simulates the body falling under gravity, watching the DOF relax to its equilibrium while the body translates.


In [ ]:
import numpy as np

import softmobility as sm
from softmobility.classes import figstyle

figstyle.apply()
np.set_printoptions(precision=3, suppress=True)

## 1. A one-DOF deformable dumbbell

Two unit-density spheres on either side of the body origin.
The bending DOF `theta0` rotates each sphere antisymmetrically around $\boldsymbol{E}_2$, and a torsion spring of stiffness `k` provides a linear restoring torque.


In [ ]:
yaml_body = """
dof_names:    [theta]
design_names: [k, length]
input_names:  [gravity]

defaults:
  theta0: 0.5    # initial tilt (radians)
  k:      5.0
  length: 2.0
  mass:   1.0

spheres:
  - radius: 0.5
    position:    [-length/2, 0, 0]
    orientation: [0, theta0, 0]
    force:       [mass * gravity0, mass * gravity1, mass * gravity2]
    torque:      [0, -k * theta0, 0]
  - radius: 0.5
    position:    [length, 0, 0]
    orientation: [0, -theta0, 0]
    force:       [mass * gravity0, mass * gravity1, mass * gravity2]
    torque:      [0, k * theta0, 0]
"""

body = sm.SoftBody(yaml_body, verbose=False)
print(repr(body))

## 2. The soft mobility tensors

`compute_tensors()` returns a named tuple with five blocks.
Their shapes follow the body dimensions: `Nspheres`, `Ndof`, `Ninput`.


In [ ]:
tensors = body.compute_tensors()
print("Shapes:")
print("  M       (soft mobility tensor) :", tensors.M.shape)
print("  M_K     (DOFs -> unknowns)     :", tensors.M_K.shape)
print("  M_H     (inputs -> unknowns)   :", tensors.M_H.shape)
print("  C_E     (strain coupling)      :", tensors.C_E.shape)
print("  Pi      (projector )           :", tensors.Pi.shape)
print()
print("M_H (gravity sensitivity):\n", tensors.M_H)
print()
print("M_K (DOF sensitivity):\n", tensors.M_K)

## 3. Simulate a trajectory under gravity

Gravity along $-\boldsymbol{e}_3$ (pulls the body down).
The DOF starts at its default value `0.5` rad of bend; the spring drives it back toward `0`.


In [ ]:
rollout = sm.FlowBodyRollout(
    soft_body=body,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=1)},
)

dt = 0.01
n_steps = 600
positions, orientations, dofs = rollout.rollout(dt=dt, n_steps=n_steps)
positions.block_until_ready()
print("positions shape:", positions.shape)
print("dofs shape     :", dofs.shape)
print("final dof      :", dofs[-1])
print("final z        :", float(positions[-1, 2]))

## 4. Plot the trajectory and the DOF

The body falls along $-\boldsymbol{e}_3$ while the spring relaxes the bending DOF back toward zero.
The two motions are **coupled**: the soft mobility tensor `M_H` carries the gravity force into a small DOF rate, and `M_K` carries the elastic restoring torque (proportional to `theta0`) into both DOF relaxation and a transient body translation.


In [ ]:
t = np.arange(n_steps) * dt

fig, axes = figstyle.subplots(size="full", aspect=2.4, ncols=2)
axes[0].plot(t, np.asarray(positions[:, 2]), color=figstyle.COLORS["blue"], linewidth=1.5)
axes[0].set_xlabel("time")
axes[0].set_ylabel("z(t)")
axes[0].set_title("Body z-position")

axes[1].plot(t, np.asarray(dofs[:, 0]), color=figstyle.COLORS["red"], linewidth=1.5)
axes[1].set_xlabel("time")
axes[1].set_ylabel(r"$\theta_0(t)$  [rad]")
axes[1].set_title("Bending DOF $\\theta_0$")

## Notes

* `compute_tensors()` is a **single forward evaluation** at the supplied `dofs`, `design`, and `time`.
  The full nonlinear hydrodynamic problem is reduced once at every time step inside the rollout.
* The tensors are JAX-compatible — `jax.jit`, `jax.grad`, and `jax.vmap` all work, so derivatives with respect to design parameters or initial conditions are available for free.
  Tutorial **04 — optimization** uses this for gradient-based design.
* For multi-DOF bodies (swimmers, fibers), the same workflow applies — the tensor shapes simply grow with `Ndof`.